In [45]:
# load data from tsp file
import numpy as np

tsp_data = np.loadtxt('gr17.2085.tsp')#fri26_d.937 
print(tsp_data.shape)

(17, 17)


In [46]:
import math
import queue
from collections import Counter

def initial_solution(length):
    s = np.random.choice(range(length), length, replace=False) #cities in [0,length]
    return s
    
def objectivefunction(s):
    cost = 0
    for i in range(s.shape[0]-1):
        cost = cost + tsp_data[s[i]][s[i+1]]
    
    cost = cost + tsp_data[s[-1]][s[0]]
    return cost

def initial_population(size, length):
    population = [] # set initially to empty
    
    # complete code here
    for i in range(size):
        # La solution initiale est une permutation random des villes
        solution = np.random.permutation(length)
        population.append(solution)
        
    return population

def evaluate_population(pop):
    
    cost_pop = []
    
    # complete code here
    for i in pop:
        # Le coût est donné par l'objectivefunction et on créé un tableau des coûts
        cost = objectivefunction(i)
        cost_pop.append(cost)
        
    return cost_pop   

def tournament_selection (pop, cost_pop, k): # k is the size of selection
    
    # complete code here
    # On choisit des individus aléatoirement et on prend le meilleur
    random_select = np.random.choice(len(pop), k , replace=False)
    for i in random_select:
        best = random_select[np.argmin(cost_pop[i])]
    
    return pop[best] # best indiv among k ones picked randomly from pop

def roulettewheel_selection (pop, cost_pop): 
    
    pop_prob = cost_pop/sum(cost_pop) #divide each cost by the sum of the costs to get a prob
    
    # complete code here
    Fitness = np.max(cost_pop) - cost_pop
    proba = Fitness/np.sum(Fitness) # On calcule la probabilité de chaque individu
    
    choice = pop[np.random.choice(range(len(pop)), p=proba)] # on choisit en fonction de la probabilité
    return  choice #indiv
    

def replacement_crossover(parent,offspring,cross_pos):   
    # On prend un point gauche et un point droit pour le crossover
    left, right = cross_pos[0], cross_pos[1]
    # deal with left side
    for i in range(left):
        # Si la valeur actuelle n'est pas un doublon on continue
        if offspring[i] not in offspring[left:right+1]:
            continue

        # Ici on remplace les doublons en cherchant une valeur valide
        for j in range(left, right+1):
            if parent[j] not in offspring: # On prend un parent pas dans l'offspring
                offspring[i] = parent[j]
                break

    # deal with right side
    for i in range(right+1, len(offspring)):
        # Si la valeur actuelle n'est pas un doublon on continue
        if offspring[i] not in offspring[left:right+1]:
            continue
        
        # Ici on remplace les doublons en cherchant une valeur valide
        for j in range(left, right+1):
            if parent[j] not in offspring: # On prend un parent pas dans l'offspring
                offspring[i] = parent[j]
                break
        
    return offspring# cleaned offspring
    
    

def pmx_crossover(parent_1, parent_2):
    # pick 2 random crossver positions
    cross_pos = np.random.choice(range(1,len(parent_1)-1), 2, replace=False)
    cross_pos = np.sort(cross_pos)
    
    #print('cp {}'.format(cross_pos))
    
    offspring_1 = np.zeros(len(parent_1), dtype=np.uint8)
    offspring_1 [:cross_pos[0]] = parent_1 [0:cross_pos[0]]
    offspring_1 [cross_pos[1]:] = parent_1 [cross_pos[1]:]
    offspring_1 [cross_pos[0]:cross_pos[1]+1] = parent_2 [cross_pos[0]:cross_pos[1]+1]
    final_offspring_1 = replacement_crossover(parent_1,offspring_1,cross_pos)
    
    offspring_2 = np.zeros(len(parent_1), dtype=np.uint8)
    offspring_2 [:cross_pos[0]] = parent_2 [0:cross_pos[0]]
    offspring_2 [cross_pos[1]:] = parent_2 [cross_pos[1]:]
    offspring_2 [cross_pos[0]:cross_pos[1]+1] = parent_1 [cross_pos[0]:cross_pos[1]+1]
    final_offspring_2 = replacement_crossover(parent_2,offspring_2,cross_pos)
    
    #print('o1 {}'.format(final_offspring_1))
    #print('o2 {}'.format(final_offspring_2))
    
    return offspring_1, offspring_2

def mutation(individual):
    
    # complete code here
    i, j = np.random.choice(range(len(individual)), 2, replace=False) # On choisit 2 indices random
    individual[i], individual[j] = individual[j], individual[i] # On permute
    return individual# muted individual
        
def mutation_decision(prob): 
    flags = ['yes','no']
    decision = np.random.choice(flags, 1, p=[prob, 1-prob]) # select yes or no according to p
    if decision == 'yes':
        return True
    else:
        return False
    
def getbestparentandofs(parent_1,parent_2, ofs_1, ofs_2):
    
    # Complete code here
    # Meilleur parent selon l'objectivefunction
    if objectivefunction(parent_1) < objectivefunction(parent_2):
        best_parent = parent_1
    else:
        best_parent = parent_2
    
    # Meilleur ofs selon l'objectivefunction
    if objectivefunction(ofs_1) < objectivefunction(ofs_2):
        best_ofs = ofs_1
    else:
        best_ofs = ofs_2
    
    tab = [] # On renvoie un tableau avec le meilleur parent et le meilleur ofs
    tab.append(best_parent)
    tab.append(best_ofs)

    return tab
    
    # best parent among the two and best ofs among the two

In [47]:
def ga(): # genetic algoithm function
    
    pop_size = 200 # must be a power of 2 for code optimization
    individual_size = 17 # if your tsp is 17 cities
    max_generations = 100
    mutation_prob = 0.3
    tournament_size = 8
    elitism_size = 40 # must be a power of 2 for code optimization 
    steady = False

    pop = initial_population(pop_size,individual_size)
    cost_pop = evaluate_population(pop)

    generations = 0
    while (generations < max_generations):
    
        intermediate_pop = []
    
        # prepare elitism
        indices = np.argpartition(cost_pop, elitism_size)[:elitism_size]
        intermediate_pop = [pop[indices[index]] for index in range (elitism_size)] # keep n best individuals
    
        ################ reproduction ################
        for _ in range ((pop_size - elitism_size)//2):
            
            # select tow parents
            parent_1 = tournament_selection(pop, cost_pop, tournament_size)
            parent_2 = tournament_selection(pop, cost_pop, tournament_size)
        
            # crossover
            ofs_1, ofs_2 = pmx_crossover(parent_1, parent_2)
    
            # mutation with probability
            if (mutation_decision (mutation_prob)):
                ofs_1 = mutation(ofs_1)
            if (mutation_decision (mutation_prob)):
                ofs_2 = mutation(ofs_2)
                
            # steady state replacement  
            if steady:
                ind_1, ind_2 = getbestparentandofs(parent_1,parent_2, ofs_1, ofs_2)
            else:
                ind_1, ind_2 = ofs_1, ofs_2
        
            intermediate_pop.append(ind_1)
            intermediate_pop.append(ind_2)
    
        # replacement of the population
        pop = np.copy(intermediate_pop)
        np.random.shuffle(pop)
    
        # evaluation 
        cost_pop = evaluate_population(pop)
    
        generations+=1
    
    
    index_best = np.array(cost_pop).argmin()
    return cost_pop[index_best], pop[index_best]

In [48]:
print(ga())

(2085.0, array([ 5, 16, 13, 14,  2, 10,  9,  1,  4,  8, 11, 15,  0,  3, 12,  6,  7]))
